In [25]:
import pandas as pd
from pathlib import Path

In [26]:
data = pd.read_csv(r"C:\shared\LSBU\ablation-study-with-developed-hypertuned-models-for-anomaly-detection-in-hrc-robotics\combined_dataset.csv")

print("rows   :", len(data))
print("columns:", data.shape[1])

rows   : 1556661
columns: 122


In [27]:
# run_number tells us which recording each row came from. It is bookkeeping,
# kept it because the validation split needs it, and add it back at the end.
bookkeeping = ["run_number"]

sensors = [c for c in data.columns if c not in bookkeeping]

print("sensor channels:", len(sensors))

sensor channels: 121


In [28]:
# Identifying the features with the same 1.5 million rows
variation = data[sensors].std()
constant = variation[variation == 0]

print("constant features:", len(constant))
print(list(constant.index))

constant features: 12
['actual_digital_input_bits', 'actual_digital_output_bits', 'robot_mode', 'joint_mode_0', 'joint_mode_1', 'joint_mode_2', 'joint_mode_3', 'joint_mode_4', 'joint_mode_5', 'safety_mode', 'target_speed_fraction', 'runtime_state']


In [29]:
# removing the features that are constant
sensors = [c for c in sensors if c not in constant.index]

print("features left:", len(sensors))

features left: 109


In [30]:
# Some channels are duplicates
correlation = data[sensors].corr().abs()

pairs = []

for a in sensors:
    for b in sensors:

        # a < b compares the names alphabetically. This does two jobs at once:
        # it stops a column being compared with itself, and it sees each pair
        # only once instead of both A-B and B-A.
        if a < b and correlation.loc[a, b] > 0.9999:
            pairs.append({"column A": a, "column B": b, "correlation": correlation.loc[a, b]})

pairs = pd.DataFrame(pairs).sort_values("correlation", ascending=False)

print("pairs found:", len(pairs))
pairs

pairs found: 19


,column A,column B,correlation
18,joint_control_output_5,target_current_5,1.000000
16,joint_control_output_3,target_current_3,1.000000
15,joint_control_output_2,target_current_2,1.000000
13,joint_control_output_0,target_current_0,1.000000
14,joint_control_output_1,target_current_1,1.000000
17,joint_control_output_4,target_current_4,1.000000
0,actual_q_0,target_q_0,1.000000
7,actual_TCP_pose_0,target_TCP_pose_0,1.000000
5,actual_q_5,target_q_5,1.000000
8,actual_TCP_pose_1,target_TCP_pose_1,1.000000


In [31]:
# Only pairs that round to 1.000000 are treated as duplicates.
# Where a pair is a command and a measurement, we keep the measurement.

# to_dict("records") turns the table into a plain list of dictionaries,
# one dictionary per row, so pair["correlation"] is a normal lookup.
to_drop = []

for pair in pairs.to_dict("records"):

    if round(pair["correlation"], 6) == 1.0:

        if pair["column A"].startswith("target"):
            to_drop.append(pair["column A"])
        else:
            to_drop.append(pair["column B"])

# set() removes any column named twice; sorted() puts them in a readable order
to_drop = sorted(set(to_drop))

print("dropping", len(to_drop), "columns:")
for name in to_drop:
    print("   ", name)

sensors = [c for c in sensors if c not in to_drop]

print()
print("channels left:", len(sensors))

dropping 13 columns:
    target_TCP_pose_0
    target_TCP_pose_1
    target_current_0
    target_current_1
    target_current_2
    target_current_3
    target_current_4
    target_current_5
    target_q_0
    target_q_1
    target_q_2
    target_q_3
    target_q_5

channels left: 96


In [32]:
# time_in_node feature is created here which means how many seconds we have been inside the current program step.
# It resets at every new step, and also at every new run.
# It is the only feature that can detect the long_wait fault: during that fault

node = data["output_double_register_21"]
run  = data["run_number"]
clock = data["timestamp"]

# node.shift() moves the whole column down by one row, so each row can see the value from the row above it.
# node != node.shift() is True wherever the node number differs from the row above, the moment a new program step begins.
# run != run.shift() does the same for the run_number.
# | means OR. So new_step is True if either a new step began or a new recording began.
new_step = (node != node.shift()) | (run != run.shift())

# .cumsum() adds up as it goes. On True/False values, True counts as 1 and False as 0.
step_number = new_step.cumsum()

data["time_in_node"] = clock - clock.groupby(step_number).transform("first")

sensors.append("time_in_node")

print(data["time_in_node"].describe())

count    1.556661e+06
mean     8.591187e-01
std      1.006256e+00
min      0.000000e+00
25%      2.600000e-01
50%      5.700000e-01
75%      1.140000e+00
max      9.970000e+00
Name: time_in_node, dtype: float64


C:\Users\serio\AppData\Local\Temp\ipykernel_2924\2877178174.py:18: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data["time_in_node"] = clock - clock.groupby(step_number).transform("first")


In [33]:
remove = [
    # The controller's clock. It counts up forever and never repeats
    "timestamp",

    # The dataset authors' run counter. Not a measurement.
    "output_double_register_20",

    # The dial that creates the scale_speed fault.
    "speed_scaling",
]

sensors = [c for c in sensors if c not in remove]

print("features:", len(sensors))

features: 94


In [34]:
for name in sensors:
    print(name)

target_q_4
target_qd_0
target_qd_1
target_qd_2
target_qd_3
target_qd_4
target_qd_5
target_qdd_0
target_qdd_1
target_qdd_2
target_qdd_3
target_qdd_4
target_qdd_5
target_moment_0
target_moment_1
target_moment_2
target_moment_3
target_moment_4
target_moment_5
target_TCP_pose_2
target_TCP_pose_3
target_TCP_pose_4
target_TCP_pose_5
target_TCP_speed_0
target_TCP_speed_1
target_TCP_speed_2
target_TCP_speed_3
target_TCP_speed_4
target_TCP_speed_5
actual_execution_time
actual_q_0
actual_q_1
actual_q_2
actual_q_3
actual_q_4
actual_q_5
actual_qd_0
actual_qd_1
actual_qd_2
actual_qd_3
actual_qd_4
actual_qd_5
actual_current_0
actual_current_1
actual_current_2
actual_current_3
actual_current_4
actual_current_5
joint_temperatures_0
joint_temperatures_1
joint_temperatures_2
joint_temperatures_3
joint_temperatures_4
joint_temperatures_5
actual_TCP_pose_0
actual_TCP_pose_1
actual_TCP_pose_2
actual_TCP_pose_3
actual_TCP_pose_4
actual_TCP_pose_5
actual_TCP_speed_0
actual_TCP_speed_1
actual_TCP_speed_2
actu

In [35]:
train_dataset = data[sensors + ["run_number"]]

print("rows   :", len(train_dataset))
print("columns:", train_dataset.shape[1])

train_dataset.to_csv(r"C:\shared\LSBU\ablation-study-with-developed-hypertuned-models-for-anomaly-detection-in-hrc-robotics\train_dataset.csv", index=False)

print("saved")

rows   : 1556661
columns: 95
saved


In [46]:
# load the test data file
test = pd.read_csv(r"C:\shared\LSBU\ablation-study-with-developed-hypertuned-models-for-anomaly-detection-in-hrc-robotics\combined_test_dataset.csv")

print("rows   :", len(test))
print("columns:", test.shape[1])

rows   : 246151
columns: 124


In [48]:
# time_in_node feature for the test dataset

node = test["output_double_register_21"]
run  = test["run_number"]
clock = test["timestamp"]

new_step = (node != node.shift()) | (run != run.shift())

step_number = new_step.cumsum()

test["time_in_node"] = clock - clock.groupby(step_number).transform("first")

print(test["time_in_node"].describe())

count    246151.000000
mean          1.447210
std           2.444488
min           0.000000
25%           0.300000
50%           0.670000
75%           1.490000
max          19.990000
Name: time_in_node, dtype: float64


In [50]:
# label and anomaly_type travel with the file so the evaluation can use them later.
# The model never sees them. It only ever sees the columns listed in `sensors`.
columns_to_keep = sensors + ["run_number", "label", "anomaly_type"]

test_dataset = test[columns_to_keep]

print("rows   :", len(test_dataset))
print("columns:", test_dataset.shape[1])

test_dataset.to_csv(r"C:\shared\LSBU\ablation-study-with-developed-hypertuned-models-for-anomaly-detection-in-hrc-robotics\test_dataset.csv", index=False)

print("saved")

rows   : 246151
columns: 97
saved
